<a href="https://colab.research.google.com/github/DAGP1145/Mineriadedatos/blob/Control-de-cambios/Mineria_De_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Fuente: https://exoplanetarchive.ipac.caltech.edu/index.html

Disponemos de muchas fuentes de informaciòn de las cuales teniamos dos principales opciones:

Sistema planetarios o sistemas planetarios compuestos.

La diferencia es que uno esta compuesto de un estudio en especifico por lo tanto lo datos son consistentes. Pero la desventaja es que podrian faltar datos porque simplemente ese estudio no midiò todo.

Por otro lado la segunda tabla esta compuesta de distintas publicaciones y las junta para crear una sola fila por planeta, por lo tanto es mas completa. El problema es que esa información puede no ser consistente, porque viene de distintos estudios que usaron métodos diferentes.

Se seleccionó la tabla de Parámetros Compuestos, ya que proporciona una única fila por planeta con un conjunto de datos más completo, lo que facilita el análisis y la aplicación de técnicas de minería de datos, aun cuando los datos no sean completamente consistentes entre sí.

In [ ]:
import pandas as pd # Estructura y manipulación de datos
import numpy as np #Programación vectorial

# Carga de los datos

In [ ]:
url = 'https://raw.githubusercontent.com/DAGP1145/Mineriadedatos/refs/heads/main/PSCompPars_2026.04.21_08.17.48.csv'
df = pd.read_csv(url, sep=",", comment="#")

In [19]:
display(df.head())

,pl_name,hostname,sy_snum,sy_pnum,sy_mnum,discoverymethod,disc_year,disc_locale,disc_facility,disc_telescope,...,st_loggerr1,st_loggerr2,st_logglim,st_dens,st_denserr1,st_denserr2,st_denslim,sy_dist,sy_disterr1,sy_disterr2
0,11 Com b,11 Com,2,1,0,Radial Velocity,2007.0,Ground,Xinglong Station,2.16 m Telescope,...,0.08,-0.08,0.0,NaN,NaN,NaN,NaN,93.1846,1.9238,-1.9238
1,11 UMi b,11 UMi,1,1,0,Radial Velocity,2009.0,Ground,Thueringer Landessternwarte Tautenburg,2.0 m Alfred Jensch Telescope,...,0.07,-0.07,0.0,NaN,NaN,NaN,NaN,125.3210,1.9765,-1.9765
2,14 And b,14 And,1,1,0,Radial Velocity,2008.0,Ground,Okayama Astrophysical Observatory,1.88 m Telescope,...,0.06,-0.07,0.0,NaN,NaN,NaN,NaN,75.4392,0.7140,-0.7140
3,14 Her b,14 Her,1,2,0,Radial Velocity,2002.0,Ground,W. M. Keck Observatory,10 m Keck I Telescope,...,0.02,-0.02,0.0,1.273928,0.349533,-0.30728,0.0,17.9323,0.0073,-0.0073
4,16 Cyg B b,16 Cyg B,3,1,0,Radial Velocity,1996.0,Ground,Multiple Observatories,Multiple Telescopes,...,0.01,-0.01,0.0,1.011029,0.292731,-0.20081,0.0,21.1397,0.0110,-0.0111


# Exploración inicial de los datos.

Seleccionaremos las columnas que queremos utilizar, esto lo hacemos porque se descargaron columnas adicionales
al Dataframe original. Porque para un mismo valor existen valores de referencias que se promedian para obtener el valor original.

Ejemplo:

st_dens: es el dato que representa la densidad del sol a estudiar pero para ese mismo dato pueden haber otros de distintos estudios.

Por ejemplo:
st_denserr1 y
st_denserr2

In [ ]:
systemplanets = df[['pl_name', 'hostname', 'sy_snum', 'sy_pnum', 'sy_mnum', 'discoverymethod', 'disc_year', 'disc_facility', 'disc_telescope', 'disc_instrument', 'pl_orbper', 'pl_rade', 'pl_radj', 'pl_dens', 'pl_orbeccen', 'pl_eqt', 'st_spectype', 'st_teff', 'st_rad', 'st_mass', 'st_met', 'st_logg', 'sy_dist']]

In [ ]:
systemplanets.head()

,pl_name,hostname,sy_snum,sy_pnum,sy_mnum,discoverymethod,disc_year,disc_facility,disc_telescope,disc_instrument,...,pl_dens,pl_orbeccen,pl_eqt,st_spectype,st_teff,st_rad,st_mass,st_met,st_logg,sy_dist
0,11 Com b,11 Com,2,1,0,Radial Velocity,2007.0,Xinglong Station,2.16 m Telescope,Coude Echelle Spectrograph,...,14.90,0.2380,NaN,G8 III,4874.0,13.76,2.09,-0.26,2.45,93.1846
1,11 UMi b,11 UMi,1,1,0,Radial Velocity,2009.0,Thueringer Landessternwarte Tautenburg,2.0 m Alfred Jensch Telescope,Coude Echelle Spectrograph,...,13.80,0.0800,NaN,K4 III,4213.0,29.79,2.78,-0.02,1.93,125.3210
2,14 And b,14 And,1,1,0,Radial Velocity,2008.0,Okayama Astrophysical Observatory,1.88 m Telescope,HIDES Echelle Spectrograph,...,2.76,0.0000,NaN,K0 III,4888.0,11.55,1.78,-0.21,2.55,75.4392
3,14 Her b,14 Her,1,2,0,Radial Velocity,2002.0,W. M. Keck Observatory,10 m Keck I Telescope,HIRES Spectrometer,...,7.96,0.3683,NaN,K0,5338.0,0.93,0.97,0.43,4.45,17.9323
4,16 Cyg B b,16 Cyg B,3,1,0,Radial Velocity,1996.0,Multiple Observatories,Multiple Telescopes,Multiple Instruments,...,1.26,0.6800,NaN,G3 V,5750.0,1.13,1.08,0.06,4.36,21.1397


# Tamaño del dataset

In [21]:
systemplanets.shape

(6160, 23)

df.shape para conocer las dimensiones de la base de datos. Manejamos 6160 filas con 23 columnas.

# Tipos de datos de las columnas

Con info() Podremos visualizar las columnas, los tipos de datos, ademas de contabilizarlos.

In [20]:
systemplanets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6160 entries, 0 to 6159
Data columns (total 23 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   pl_name          6160 non-null   object 
 1   hostname         6160 non-null   object 
 2   sy_snum          6160 non-null   int64  
 3   sy_pnum          6160 non-null   int64  
 4   sy_mnum          6160 non-null   int64  
 5   discoverymethod  6160 non-null   object 
 6   disc_year        6159 non-null   float64
 7   disc_facility    6160 non-null   object 
 8   disc_telescope   6160 non-null   object 
 9   disc_instrument  6160 non-null   object 
 10  pl_orbper        5821 non-null   float64
 11  pl_rade          6110 non-null   float64
 12  pl_radj          6110 non-null   float64
 13  pl_dens          6021 non-null   float64
 14  pl_orbeccen      5221 non-null   float64
 15  pl_eqt           4596 non-null   float64
 16  st_spectype      2314 non-null   object 
 17  st_teff       

Podriamos usar
df.columns para conocer las columnas. Tambien podriamos utilizar df.dtypes para conocer los tipos de datos de cada columa.

Pero con info es mas que suficiente para conocer ambas datos, además nos permite contabilizar los tipos de datos función que no hace con dtypes.

Una alternativa en tal caso de que no se este del todo conforme con lo mencionado puede ser: df.dtypes.value_counts()

Pero no visualizamos cuáles son las columnas en su conjunto que pertenecen a cierto tipo de dato. A continuación lo hacemos.

In [26]:
df_objetos = systemplanets.select_dtypes(include=['object'])
df_int_floats = systemplanets.select_dtypes(include=['int', 'float'])

In [54]:
display(df_objetos.columns.to_list())
print("Numero de columnas: ", sum(df_objetos.columns.value_counts()), '\n')
display(df_int_floats.columns.to_list())
print("Numero de columnas: ", sum(df_int_floats.columns.value_counts()))

['pl_name',
 'hostname',
 'discoverymethod',
 'disc_facility',
 'disc_telescope',
 'disc_instrument',
 'st_spectype']

Numero de columnas:  7 



['sy_snum',
 'sy_pnum',
 'sy_mnum',
 'disc_year',
 'pl_orbper',
 'pl_rade',
 'pl_radj',
 'pl_dens',
 'pl_orbeccen',
 'pl_eqt',
 'st_teff',
 'st_rad',
 'st_mass',
 'st_met',
 'st_logg',
 'sy_dist']

Numero de columnas:  16


Por lo tanto existen en total 7 columnas categoricas y para las columnas que componen los datos numericos corresponden a un total de 16. Conformando un total de 23 de columnas dentro de nuestro dataset.

# Tratamiento de valores nulos

In [55]:
for feature in systemplanets.columns:
  print('Total de valores nulos de', feature, '=', systemplanets[feature].isna().sum())

Total de valores nulos de pl_name = 0
Total de valores nulos de hostname = 0
Total de valores nulos de sy_snum = 0
Total de valores nulos de sy_pnum = 0
Total de valores nulos de sy_mnum = 0
Total de valores nulos de discoverymethod = 0
Total de valores nulos de disc_year = 1
Total de valores nulos de disc_facility = 0
Total de valores nulos de disc_telescope = 0
Total de valores nulos de disc_instrument = 0
Total de valores nulos de pl_orbper = 339
Total de valores nulos de pl_rade = 50
Total de valores nulos de pl_radj = 50
Total de valores nulos de pl_dens = 139
Total de valores nulos de pl_orbeccen = 939
Total de valores nulos de pl_eqt = 1564
Total de valores nulos de st_spectype = 3846
Total de valores nulos de st_teff = 294
Total de valores nulos de st_rad = 318
Total de valores nulos de st_mass = 8
Total de valores nulos de st_met = 554
Total de valores nulos de st_logg = 322
Total de valores nulos de sy_dist = 27


Para el tratamiento de nuos visualizamos una considerable cantidad en algunas columnas:

disc_year = 1. Corresponde al año de descubrimiento.
pl_orbper = 339. Corresponde al periodo en dias en que orbita el planeta a su Sol.
pl_rade = 50. Corresponde al radio de planeta medido en radio de la tierra.
pl_radj = 50. Lo mismo pero con jupiter.
pl_dens = 139. Corresponde a la densidad del planeta.
pl_orbeccen = 939
pl_eqt = 1564
st_spectype = 3846
st_teff = 294
st_rad = 318
st_mass = 8
st_met = 554
st_logg = 322
sy_dist = 27

